# BuildMate Rentals — Gold Layer
**Task 3 · Build the conformed star and tune it**

Nothing below exists in any source system — we build the dimensions and facts here.

**Acceptance criteria**
- `gold_dim_customer` 600, `gold_dim_depot` 6, `gold_dim_date` 30,
  `gold_fact_rental` 942, `gold_fact_billing` 779
- `rental_duration_days` is null for exactly the still-out rentals; `is_returned` = 0 for them
- no `asset_days_in_window` value exceeds 30
- `OPTIMIZE ... ZORDER` has run and `DESCRIBE HISTORY` shows it


In [ ]:
from pyspark.sql import functions as F

WIN_START = "2026-06-01"
WIN_END   = "2026-06-30"   # inclusive 30-day reporting window

silver_customers = spark.table("silver_customers")
silver_depots    = spark.table("silver_depots")
silver_rentals   = spark.table("silver_rentals")
silver_billing   = spark.table("silver_billing")

## 3.1 `gold_dim_customer` — derive tenure (not in any source)
`tenure_years` and `tenure_band` are computed against the window end so the numbers are
reproducible run-to-run.

In [ ]:
ref = F.to_date(F.lit(WIN_END))
tenure_years = F.floor(F.datediff(ref, F.col("registered_on")) / 365.25)
tenure_band = (F.when(tenure_years < 1, "New (<1y)")
                .when(tenure_years < 3, "Growing (1-3y)")
                .when(tenure_years < 5, "Established (3-5y)")
                .otherwise("Veteran (5y+)"))

gold_dim_customer = (silver_customers
    .withColumn("tenure_years", tenure_years.cast("int"))
    .withColumn("tenure_band", tenure_band)
    .select("customer_id","customer_name","customer_type","city",
            "registered_on","kyc_verified_on","tenure_years","tenure_band"))
gold_dim_customer.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable("gold_dim_customer")
print("gold_dim_customer:", spark.table("gold_dim_customer").count())   # 600
spark.table("gold_dim_customer").groupBy("tenure_band").count().show()

## 3.2 `gold_dim_depot` — already clean, promote as-is

In [ ]:
spark.table("silver_depots").write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable("gold_dim_depot")
print("gold_dim_depot:", spark.table("gold_dim_depot").count())   # 6

## 3.3 `gold_dim_date` — generate the 30-day calendar (no source will ever send one)

In [ ]:
gold_dim_date = spark.sql(f'''
    SELECT
        d                                        AS date,
        year(d)                                  AS year,
        month(d)                                 AS month,
        day(d)                                   AS day_of_month,
        date_format(d, 'EEEE')                   AS day_name,
        weekday(d) >= 5                          AS is_weekend
    FROM (SELECT explode(sequence(to_date('{WIN_START}'), to_date('{WIN_END}'), interval 1 day)) AS d)
''')
gold_dim_date.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable("gold_dim_date")
print("gold_dim_date:", spark.table("gold_dim_date").count())   # 30

## 3.4 `gold_fact_rental` — keep every rental, returned or not
- `rental_duration_days`: **NULL** while the machine is still out — that is the honest answer,
  not zero. A zero would say "returned same day," which is false.
- `asset_days_in_window`: the rented days that fall *inside* the window, clipped with
  `greatest`/`least` so a long rental is never counted beyond the window edge (max 30).

In [ ]:
co = F.to_date("checkout_ts")
ci = F.to_date("checkin_ts")
ws = F.to_date(F.lit(WIN_START))
we = F.to_date(F.lit(WIN_END))

start = F.greatest(co, ws)
end   = F.least(F.coalesce(ci, we), we)              # still-out -> counted up to window edge
asset_days = F.greatest(F.lit(0), F.datediff(end, start) + F.lit(1))   # inclusive; full window = 30

gold_fact_rental = (silver_rentals
    .withColumn("asset_type", F.regexp_extract("asset_id", r"^([A-Z]+)-", 1))
    .withColumn("is_returned", F.col("checkin_ts").isNotNull().cast("int"))
    .withColumn("rental_duration_days",
                F.when(F.col("checkin_ts").isNotNull(), F.datediff(ci, co)))   # else NULL
    .withColumn("asset_days_in_window", asset_days)
    .select("rental_id","customer_id","depot_code","asset_id","asset_type",
            "checkout_ts","checkin_ts","rental_type",
            "is_returned","rental_duration_days","asset_days_in_window"))
gold_fact_rental.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable("gold_fact_rental")

f = spark.table("gold_fact_rental")
print("gold_fact_rental:", f.count())                                              # 942
print("still-out (is_returned=0):", f.filter("is_returned = 0").count())           # 175
print("duration NULL rows       :", f.filter("rental_duration_days IS NULL").count())  # 175 (== still-out)
print("max asset_days_in_window :", f.agg(F.max("asset_days_in_window")).first()[0])   # <= 30
assert f.filter("is_returned = 0 AND rental_duration_days IS NOT NULL").count() == 0
assert f.filter("asset_days_in_window > 30").count() == 0
print("Gold fact_rental asserts passed.")

## 3.5 `gold_fact_billing` — the parsed bills

In [ ]:
spark.table("silver_billing").write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable("gold_fact_billing")
print("gold_fact_billing:", spark.table("gold_fact_billing").count())   # 779

## 3.6 Tune the facts — `OPTIMIZE ... ZORDER` + `DESCRIBE HISTORY`
**Why no partitioning?** These tables are hundreds/thousands of rows. Partitioning earns its
place at hundreds of millions of rows; here it would create tiny files and *hurt* performance.
Z-ordering co-locates the columns we filter/join on within the existing files, which is the right
tool at this scale.

In [ ]:
spark.sql("OPTIMIZE gold_fact_rental  ZORDER BY (depot_code, asset_type)")
spark.sql("OPTIMIZE gold_fact_billing ZORDER BY (rental_id)")
spark.sql("DESCRIBE HISTORY gold_fact_rental").select("version","timestamp","operation").show(truncate=False)